# LLM Evaluation with DeepEval

**Intro to DeepEval · Important metrics · RAG evaluation · Custom evals: G-Eval, DAG, Arena G-Eval**

---

### The idea in one line

> **DeepEval is unit testing, but for LLM output.**

You already trust tests for normal code:

```
   assert add(2, 2) == 4
```

You cannot write that for an LLM, because there are many correct answers.
So DeepEval replaces the exact match with a **judged score and a pass mark**:

```
   assert answer_relevancy(answer) >= 0.7
```

Same habit, same safety net, adapted to text that is never identical twice.

---
# Part 1 - Introduction to DeepEval and LLM Evaluation

### Why you cannot test an LLM the normal way

| Normal code | LLM output |
|---|---|
| One correct answer | Many correct answers, worded differently |
| Same input gives same output | Same input gives a different output each run |
| Fails loudly with an error | Fails quietly with a confident wrong answer |
| `assert x == y` works | `assert x == y` is useless |

### What DeepEval gives you

1. A **test case**: what went in, what came out, and what should have come out
2. A **metric**: something that scores that test case from 0 to 1
3. A **threshold**: the pass mark, so a score becomes a pass or a fail

```
                         +-----------------------------+
   LLMTestCase  ------>  |          Metric             |  ------> score 0.86
   input                 |  (an LLM judges the output) |          threshold 0.7
   actual_output         +-----------------------------+          PASSED
   expected_output
   retrieval_context
```

### DeepEval vs RAGAS


| | RAGAS | DeepEval |
|---|---|---|
| Shape | Score a **dataset**, get a table | Score **test cases**, get pass or fail |
| Feels like | A report card | A test suite |
| Custom metrics | Limited | **Its main strength**: G-Eval, DAG, Arena |
| Best at | Measuring a RAG pipeline | Testing any LLM behaviour, plus RAG |

**Use both.** RAGAS to measure your RAG quality, DeepEval to assert that specific
behaviours hold before you ship.

### What it costs

The judge is an LLM, so **every metric on every test case is at least one API call**,
and asking for a `reason` adds another.

**DeepEval's default judge is an expensive model.** We pin `gpt-4o-mini` on every metric
in this notebook, which is roughly 15 times cheaper. Keep test sets small while learning.

In [2]:
import deepeval

print("success")

success


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()      

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"      

JUDGE = "gpt-4o-mini"

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY loaded")
else:
    print("OPENAI_API_KEY MISSING, add it to your .env file")

print("Judge model:", JUDGE)

OPENAI_API_KEY loaded
Judge model: gpt-4o-mini


### The test case: the one object everything uses

Every metric reads from an `LLMTestCase`. These are the fields that matter:

| Field | What it holds | Who fills it |
|---|---|---|
| `input` | the question | you |
| `actual_output` | what your app answered | your app |
| `expected_output` | the correct answer | you, in advance |
| `retrieval_context` | the documents your **retriever** found | your app |
| `context` | the documents that **should** have been found | you |

> **`retrieval_context` and `context` are not the same thing**, and mixing them up is the
> most common DeepEval mistake.
> `retrieval_context` is what your system really fetched.
> `context` is the ideal ground truth. Most RAG metrics want `retrieval_context`.

In [5]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input = "How long do refunds take?",
    actual_output = "Refunds are processed within 7 business days",
    expected_output = "Refunds take 7 business days",
    retrieval_context = ["Refunds are processed within 7 business days for orders under 30 days old."]
)

print("input:", test_case.input)
print("output:", test_case.actual_output)
print("retrieval_context:", test_case.retrieval_context)


input: How long do refunds take?
output: Refunds are processed within 7 business days
retrieval_context: ['Refunds are processed within 7 business days for orders under 30 days old.']


---
# Part 2 - Important DeepEval Metrics

### The two families

| Family | Question it answers | Metrics |
|---|---|---|
| **RAG metrics** | Is my retrieval and answering any good? | Answer relevancy, Faithfulness, Contextual precision, Contextual recall |
| **Safety metrics** | Is this output harmful or wrong? | Hallucination, Bias, Toxicity |

### What each one needs

| Metric | Measures | Needs |
|---|---|---|
| `AnswerRelevancyMetric` | Does the answer address the question? | `input`, `actual_output` |
| `FaithfulnessMetric` | Is the answer supported by the retrieved documents? | `input`, `actual_output`, `retrieval_context` |
| `ContextualPrecisionMetric` | Are the useful documents ranked near the top? | plus `expected_output` |
| `ContextualRecallMetric` | Did retrieval find everything needed? | plus `expected_output` |
| `HallucinationMetric` | Does the answer contradict the given context? | `context` |
| `BiasMetric` | Is the answer biased? | `actual_output` |
| `ToxicityMetric` | Is the answer rude or harmful? | `actual_output` |

All return **0 to 1**, but the direction is not the same for all of them:

| Metrics | Direction | Passing means |
|---|---|---|
| The four RAG metrics | higher is better | `score >= threshold` |
| `HallucinationMetric`, `BiasMetric`, `ToxicityMetric` | **lower is better** | `score <= threshold` |

> **This catches people out.** For the safety metrics the threshold is a **maximum**,
> not a minimum. `ToxicityMetric(threshold=0.5)` means "fail if toxicity goes above 0.5".

### Threshold: turning a score into a decision

```python
AnswerRelevancyMetric(threshold=0.7, model=JUDGE)
```

Score at or above 0.7 passes, below fails. **That is what makes this a test suite
rather than a report.**

> **Always pass `model=`.** Left alone, DeepEval picks an expensive default judge.
> Every metric in this notebook pins the cheap one.

### Run one metric

In [7]:
from deepeval.metrics import AnswerRelevancyMetric

metric = AnswerRelevancyMetric(threshold=0.7,model=JUDGE)

metric.measure(test_case)

print("score :", metric.score)
print("Passed :", metric.is_successful())
print("Reason :", metric.reason)

c:\Users\THIRU\OneDrive\Desktop\ai_security\venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

score : 1.0
Passed : True
Reason : The score is 1.00 because the response directly addressed the question about refund timelines without any irrelevant statements.


In [10]:
from deepeval import evaluate
from deepeval.metrics import FaithfulnessMetric

good_case = LLMTestCase(
    input = "How long do refunds take?",
    actual_output = "Refunds are processed within 7 business days",
    retrieval_context = ["Refunds are processed within 7 business days for orders under 30 days old."]
)

bad_case = LLMTestCase(
    input = "How long do refunds take?",
    actual_output = "we refund instantly by UPI, and we also give free vouchers",
    retrieval_context = ["Refunds are processed within 7 business days for orders under 30 days old."]
)

results = evaluate(
    test_cases = [good_case, bad_case],
    metrics=[AnswerRelevancyMetric(threshold=0.7,model=JUDGE),FaithfulnessMetric(threshold=0.7,model=JUDGE)]
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

c:\Users\THIRU\OneDrive\Desktop\ai_security\venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In get_cached_test_run, temp=False, Lock acquisition failed: Shared locks on Windows require the win32 extra 
(pywin32); msvcrt provides no true shared lock. Install it with: pip install "portalocker[win32]"

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            How long do refunds take?                                                              │
│  │     Actual Output:    we refund instantly by UPI, and we also give free vouchers                             │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.50  │ 0.70      │ The score is 0.50 because the output included an          │
│              │                  │       │           │ irrelevant statement about free vouchers, which does      │
│              │                  │       │           │ not address the question of how long refunds take. This   │
│              │                  │       │           │ detracted from the overall relevance, preventing a        │
│              │                  │       │           │ higher score.                                             │
│        FAIL  │ Faithfulness     │ 0.50  │ 0.70      │ The score is 0.50 because the actual output claims        │
│              │                  │       │           │ instant refunds, which contradicts the context stating    │
│              │                  │       │           │ that refunds are processed within 7 business days.        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score          ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 0.75                   │ 50.00% | passed=1 | failed=1                 │ 2         │
│  Faithfulness               │ 0.75                   │ 50.00% | passed=1 | failed=1                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

⚠ WARNING: No hyperparameters logged.
» ]8;id=13876831;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.18s | token cost: 0.0010634999999999998 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

RAG App with Deepeval

In [11]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate

documents = [
    "Refunds are processed within 7 business days for orders placed under 30 days ago.",
    "Standard shipping takes 3 to 5 business days. Express shipping takes 1 business day.",
    "Our support team is available Monday to Friday, 9am to 6pm IST.",
    "You can cancel a subscription anytime from Settings. Billing stops at the end of the cycle.",
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_texts(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support assistant. Answer ONLY using this context: {context}"),
    ("human", "{question}"),
])

print("RAG app ready")

RAG app ready


In [12]:
questions = [
    "How long do refunds take?",
    "How fast is express shipping?",
    "When is support available?"
]

expected = [
    "Refunds are processed within 7 business days",
    "Express shipping takes 1 business days",
    "support is available monday to friday, 9am to 6pm IST."
]

test_cases = []

for question, expected_answer in zip(questions,expected):

    docs = retriever.invoke(question)

    contexts= []
    for doc in docs:
        contexts.append(doc.page_content)

    messages = prompt.format_messages(context = " ".join(contexts), question = question)

    answer = llm.invoke(messages).content

    test_cases.append(LLMTestCase(
        input = question,
        actual_output = answer,
        expected_output = expected_answer,
        retrieval_context = contexts,
    ))

    print("done", question)


done How long do refunds take?
done How fast is express shipping?
done When is support available?


In [ ]:
from deepeval.metrics import ContextualRecallMetric, ContextualPrecisionMetric

rag_metrics = [
    AnswerRelevancyMetric(threshold=0.7,model=JUDGE),
    FaithfulnessMetric(threshold=0.7,model=JUDGE),
    ContextualRecallMetric(threshold=0.7,model=JUDGE),
    ContextualPrecisionMetric(threshold=0.7,model=JUDGE)
]

results = evaluate(test_cases=test_cases, metrics=rag_metrics)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              How long do refunds take?                                                            │
│  │     Actual Output:      Refunds are processed within 7 business days for orders placed under 30 days ago.    │
│  │     Expected Output:    Refunds are processed within 7 business days                                         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.50  │ 0.70      │ The score is 0.50 because the response included           │
│              │                  │       │           │ irrelevant information about orders placed under 30       │
│              │                  │       │           │ days ago, which does not directly address the question    │
│              │                  │       │           │ of how long refunds take.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score          ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 0.83                   │ 66.67% | passed=2 | failed=1                 │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

⚠ WARNING: No hyperparameters logged.
» ]8;id=13876835;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.47s | token cost: 0.0006762 USD)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

---
# Part 4 - Custom evals

The built-in metrics cover generic quality. They cannot know **your** rules:

- "never promise a refund without mentioning the 30 day limit"
- "always answer in under 50 words"
- "must include the ticket number"

For that you write a custom metric. DeepEval gives you three tools, and choosing
between them is the real skill.

| Tool | You provide | The judge |
|---|---|---|
| **G-Eval** | a sentence describing what good looks like | decides freely |
| **DAG** | a decision tree of explicit checks | follows your tree |
| **Arena G-Eval** | two or more outputs | picks a winner |

### 4.1 - G-Eval: describe the rule in plain English

The easiest one. You write the criteria as a sentence, and an LLM scores against it.

`evaluation_params` tells it which fields to look at.

In [20]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

politeness = GEval(
    name = "politeness",
    criteria = "decide if the answer is polite and professional in tone",
    evaluation_params = [SingleTurnParams.INPUT,SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=JUDGE
)

rude_case = LLMTestCase(
    input = "where is my order?",
    actual_output = "How should i know, check it yourself."
)

politeness.measure(rude_case)

print("score :", politeness.score)
print("passed :", politeness.is_successful())
print("reason :", politeness.reason)


c:\Users\THIRU\OneDrive\Desktop\ai_security\venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

score : 0.05
passed : False
reason : The Actual Output displays a rude and dismissive tone, failing to maintain the politeness and professionalism of the Input. It lacks any polite expressions and does not provide helpful information, contrasting sharply with the neutral inquiry of the Input.


In [22]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

politeness = GEval(
    name = "politeness",
    evaluation_steps = [
        "Check whether the answer mentions refunds at all.",
        "If it does, check that it also states the 30 day limit.",
        "Penalise heavily if a refund is promised with no mention of the limit.",
    ],
    evaluation_params = [SingleTurnParams.INPUT,SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=JUDGE
)

rude_case = LLMTestCase(
    input="Can I get a refund?",
    actual_output="Yes, of course. We will refund you straight away.",
)

politeness.measure(rude_case)

print("score :", politeness.score)
print("reason :", politeness.reason)


score : 0.19274690030821615
reason : The response mentions refunds but fails to include the crucial 30-day limit, which is a significant shortcoming. Additionally, it promises an immediate refund without any conditions, which further detracts from its alignment with the evaluation criteria.


### 4.2 - DAG: a decision tree you control

G-Eval is one judgement call, so the score can wobble between runs.
A **DAG** breaks the decision into small yes or no steps that you define.

**The four building blocks:**

| Node | What it does |
|---|---|
| `BinaryJudgementNode` | asks a yes or no question. Verdicts must be `True` and `False` |
| `NonBinaryJudgementNode` | asks a multiple choice question. Verdicts are strings |
| `TaskNode` | extracts or reshapes something first, before any judging |

You build it top down with two methods:

| Method | Use on | Meaning |
|---|---|---|
| `.add_verdict(verdict=..., score=N)` | a judgement node | this answer ends here with score N |
| `.add_verdict(verdict=..., then=node)` | a judgement node | this answer continues to another question |
| `.add_node(child)` | a `TaskNode` | hand the extracted result to the next node |

The tree we are about to build:

```
                 Does it mention a number of days?
                          /              \
                       True             False
                        /                  \
        Does it say 7 business days?      score 0
              /            \
           True           False
            /                \
        score 10          score 2
```

> **Two things that trip people up.**
> 1. A `BinaryJudgementNode` needs `verdict=True` and `verdict=False`. The strings
>    `"yes"` and `"no"` raise
>    `ValueError: All children BinaryJudgementNode must have a boolean verdict.`
> 2. Older tutorials build nodes with `children=[VerdictNode(...)]`. That still works
>    but is **deprecated** and prints a warning. Use `.add_verdict()` as below.

In [23]:
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import DeepAcyclicGraph, BinaryJudgementNode


says_seven_days = BinaryJudgementNode(criteria = "Does the answer says 7 business days")
says_seven_days.add_verdict(verdict=True, score=10)
says_seven_days.add_verdict(verdict=False, score=2)

mentions_days = BinaryJudgementNode(criteria = "Does the answer mention a number of days at all?")
mentions_days.add_verdict(verdict=True, then = says_seven_days)
mentions_days.add_verdict(verdict=False, score=0)

dag = DeepAcyclicGraph(root_nodes=[mentions_days])

refund_dag = DAGMetric(name="Refund answer check", dag=dag, threshold = 0.7, model = JUDGE)

print(" DAG Metric Ready", refund_dag.name)

 DAG Metric Ready Refund answer check


In [25]:
answers = [
    "Refunds are processed within 7 business days.",     
    "Refunds usually take about 20 days.",               
    "We will sort out your refund soon.",                
]

for answer in answers:
    case = LLMTestCase(input="How long do refund take?", actual_output=answer)

    refund_dag.measure(case)

    print("Answer :", answer)
    print("passed :", refund_dag.is_successful())
    print("score :", refund_dag.score)
    

Answer : Refunds are processed within 7 business days.
passed : False
score : 0.0


Answer : Refunds usually take about 20 days.
passed : False
score : 0.0


Answer : We will sort out your refund soon.
passed : False
score : 0.0


> **Scores in a DAG are out of 10**, and DeepEval divides by 10 to give the usual
> 0 to 1 score. So `score=10` becomes 1.0 and `score=2` becomes 0.2.

> **Why bother with the tree?** Because the score is now **explainable and repeatable**.
> Anyone can read the tree and predict the score. With G-Eval you are trusting the judge's
> overall impression.

### 4.3 - Arena G-Eval: which output is better?

The other two ask *"is this good?"*. Arena asks *"which of these is better?"*

Use it to compare two prompts, two models, or two versions of your app on the same input.

**Two rules the contestants must follow:** every contestant needs a **unique name**, and
they must all share the **same `input`**. You are comparing answers to one question.

> Judging "A or B" is far more reliable than scoring each one alone, because two
> mediocre answers can both land on 0.75 and tell you nothing.

In [33]:
from deepeval.metrics import ArenaGEval
from deepeval.test_case import ArenaTestCase
from deepeval.test_case.arena_test_case import Contestant

question = "How long do refunds take?"

arena_case = ArenaTestCase(contestants=[
    Contestant(
        name="Short prompt",
        test_case=LLMTestCase(input=question, actual_output="7 business days."),
    ),
    Contestant(
        name="Detailed prompt",
        test_case=LLMTestCase(
            input=question,
            actual_output="Refunds are processed within 7 business days, "
                                    "for orders placed under 30 days ago.",
        ),
    ),
])





comparison = ArenaGEval(
    name="Most helpful answer",
    criteria="Pick the answer that is clearer and gives the customer more useful detail, "
                                    "without adding anything untrue.",
                                    
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=JUDGE,
)

comparison.measure(arena_case)

print("winner:", comparison.winner)
print("reason:", comparison.reason)

winner: Detailed prompt
reason: Detailed prompt provides a clear and detailed response, stating that refunds are processed within 7 business days for orders placed under 30 days ago. This not only answers the question but also adds context about the time frame, enhancing usefulness. In contrast, Short prompt offers a less informative answer with just '7 business days', lacking the additional detail that could help the customer understand the conditions for the refund timeline. Detailed prompt's response is both clear and informative, aligning best with the evaluation steps.


In [34]:
from deepeval import compare

win_counts = compare(test_cases=[arena_case], metric=comparison)

print(win_counts)

c:\Users\THIRU\OneDrive\Desktop\ai_security\venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

🎉 Arena completed! (time taken: 5.49s | token cost: 0.0002331 USD)
🏆 Results (1 total test cases):
    » Detailed prompt: 1 wins

================================================================================

» Want to share experiments with your team? ❤️ 🏟️
  » Run 'deepeval login' to analyze and save arena results on Confident AI.

{'Detailed prompt': 1}
